In [25]:
import yadisk # https://pypi.org/project/yadisk/
import os
from dotenv import load_dotenv

import cv2 # https://docs.opencv.org/4.x/d1/dc5/tutorial_background_subtraction.html
import numpy as np 

import pickle
import json 
import os

import ffmpeg
import pandas as pd

from tqdm import tqdm
import time

import torch
from torch.utils.data import Dataset, DataLoader


In [26]:
load_dotenv()
TOKEN = os.getenv('DEBUG_TOKEN')
client = yadisk.Client(token=TOKEN)

with client:
    # Проверяет, валиден ли токен
    print(client.check_token())

    # Выводит содержимое "disk:/SLR Project"
    files = list(client.listdir("disk:/SLR Project")) # disk:/SLR_Project_Cuts
    sub_files = list(client.listdir("disk:/SLR Project_subs"))
    print(len(files), len(sub_files))

True
482 321


In [27]:
# Субтитры
sub_path = '../load_data/id2name.json'
id2name = json.load(open(sub_path, 'r', encoding='utf-8'))
bad_symbols = '?\":/|'

caption_dict = dict() # Название видео : DataFrame(columns=["text", "start", "end"])
#new_caption_dict = dict()
for caption_file in tqdm([sf['path'] for sf in sub_files]):
    fn = 'SLR Project_subs/'+ caption_file.split('/')[-1]
    this_cap_list = []
    with open(fn, 'r', encoding='utf-8') as f:
        for caption in json.load(f):
            cap_dict = caption.copy()
            this_cap_list.append({'text': cap_dict['text'], 
                                 'start': cap_dict['start'],
                                 'end': cap_dict['start'] + cap_dict['duration']})
    
    name = caption_file.split('/')[-1].split('.')[0]
    video_name = id2name[name]+'.mp4'
    for bs in bad_symbols:
        video_name = video_name.replace(bs, '')
    if video_name == 'Интервью с двукратным сурдлимпийским чемпионом Владиславом Винником. 2 часть.mp4':
        video_name = "Интервью с двукратным сурдлимпийским чемпионом Владиславом Винником. 2 часть. С субтитрами.mp4"
    caption_dict[video_name] = pd.DataFrame(data=this_cap_list)

100%|██████████| 321/321 [00:00<00:00, 706.91it/s]


In [70]:
def find_sub_vid_overlap(sub_s, sub_e, vid_s, vid_e):
    sub = [i for i in range(int(sub_s), int(sub_e)+1, 1)]
    vid = [i for i in range(int(vid_s), int(vid_e)+1, 1)]
    overlap = len(set(sub) & set(vid))
    if overlap == len(sub):
        return True
    elif overlap > 3:
        return True
    return False

In [ ]:
def cut_the_video(path_on_disk,  path_to_store, video_name, sub_df,
                  window_size=11, overlap=5, new_fps=3):
    TOKEN = os.getenv('DEBUG_TOKEN')
    client = yadisk.Client(token=TOKEN)
    client.download(path_on_disk, path_to_store) # скачиваем файл
    cap = cv2.VideoCapture(path_to_store) # читаем файл

    size = (int(cap.get(3)), int(cap.get(4))) # ширина, высота видео
    # отсекаем видео с некошерным размером
    if size != (1280, 720):
        os.remove(path_to_store)
        return []

    frame_num = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) # frames per second
    fpw = int(fps*window_size) # frames per window 
    new_fpw = int(new_fps*window_size)
    fpo = int(fps*overlap) # frames per overlap
    step = int(fps/new_fps)
    print(fpo)

    video_json = [] # айтемы будущего датасета
    window_frame_ids = [] # в исходном разрешении

    for i in range(frame_num):        
        window_frame_ids.append(i)
        
        if len(window_frame_ids) >= fpw: # пора записывать текущее окно и обрезать
            new_data = {'video': video_name, 
                        'start': round((i-fpw)/fps, 3), 
                        'end': round(i/fps, 3),
                        'frame_ids': [], 
                        'text': ''}
            # добавляем айдишники фреймов
            for new_frame in window_frame_ids:
                if new_frame % step == 0 and len(new_data['frame_ids']) < new_fpw:
                    new_data['frame_ids'].append(new_frame)  
            # добавляем субтитры (оверлап должен быть хотя бы 5 секунд)
            frame_sub_df = sub_df[(((sub_df['start'] > new_data['start']) & 
                                    (sub_df['start'] < new_data['end'])) |
                                    ((sub_df['end'] > new_data['start']) & 
                                     (sub_df['end'] < new_data['end'])) |
                                    ((sub_df['end'] > new_data['end']) & 
                                     (sub_df['start'] < new_data['start'])) 
                                     )]
            text = ''
            for _, row in frame_sub_df.iterrows():
                if find_sub_vid_overlap(row['start'], row['end'], 
                                        new_data['start'], new_data['end']):
                    text += row['text'] + ' '
            new_data['text'] = text.strip(' ')

            video_json.append(new_data)
            window_frame_ids = window_frame_ids[fpo:] # редактируем window_framеs

        i += 1
    cap.release() # закрываем исходный файл
    os.remove(path_to_store) # удаляем исходный файл

    return video_json

In [ ]:
# Для тестов
# video_name = "#УСЛЫШЬМЕНЯ 25 сентября. Премьера по всей России. На жестовом языке, с субтитрами.mp4"
# video_name = "Итоги лыжного этапа Зимних игр-2019.mp4"
# path_on_disk = 'disk:/SLR Project/'+ video_name
# path_to_store = './' + video_name

# sub_df = caption_dict[video_name]

# ctv = cut_the_video(path_on_disk, path_to_store, video_name, sub_df,
#                 window_size=11, overlap=5, new_fps=3)
# print(len(ctv[0]['frame_ids']))
# print(ctv[0])

125
33
{'video': 'Итоги лыжного этапа Зимних игр-2019.mp4', 'start': -0.04, 'end': 10.96, 'frame_ids': [0, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256], 'text': ''}


In [19]:
df_items = []
v = 0

In [20]:
# Все видео (316 по итогу)
no_subs_set = set() # видео без субтитров нас не интересуют
with open('../EDA/no_subs.json', 'r', encoding='utf-8') as f:
    no_subs = json.load(f)
    for no_sub in no_subs:
        no_subs_set.add(no_sub['vid_path'].replace('\"', ''))

for file in tqdm(files):
    video_name = file['path'].split('/')[-1]
    if video_name not in no_subs_set:
        if video_name in caption_dict:
            v += 1
            sub_df = caption_dict[video_name]
            path_on_disk = file['path']
            path_to_store = './' + video_name
            df_item = cut_the_video(path_on_disk, path_to_store, video_name, sub_df,
                            window_size=11, overlap=5, new_fps=3)
            df_items.extend(df_item)
print(v)

  0%|          | 0/482 [00:00<?, ?it/s]

100%|██████████| 482/482 [24:10<00:00,  3.01s/it]

316


In [21]:
print(len(df_items))
with open('df_items.json', 'w', encoding='utf-8') as f:
    json.dump(df_items, f, ensure_ascii=False, indent=3)

38656


### Train / Eval / Test split

In [ ]:
import json
import random
random.seed(42)

with open('df_items.json', 'r', encoding='utf-8') as f:
    all_samples = json.load(f)

all_video_names = []
for sample in all_samples:
    if sample['video'] not in all_video_names:
        all_video_names.append(sample['video'])
    
random.shuffle(all_samples)
random.shuffle(all_video_names)

test_video_names = all_video_names[290:]
test_samples = []
train_val_samples = []
for sample in all_samples:
    if sample['video'] in test_video_names:
        test_samples.append(sample)
    else:
        train_val_samples.append(sample)
train_samples = train_val_samples[:int(len(train_val_samples)*0.8)]
val_samples = train_val_samples[int(len(train_val_samples)*0.8):]

for item in train_samples:
    item = item.update({'split': 'train'})
for item in val_samples:
    item = item.update({'split': 'eval'})
for item in test_samples:
    item = item.update({'split': 'test'})

print(train_samples[0])

with open('test.json', 'w', encoding='utf-8') as f:
    json.dump(test_samples, f, ensure_ascii=False, indent=4)
with open('train.json', 'w', encoding='utf-8') as f:
    json.dump(train_samples, f, ensure_ascii=False, indent=4)
with open('eval.json', 'w', encoding='utf-8') as f:
    json.dump(val_samples, f, ensure_ascii=False, indent=4)

print(len(train_samples))
print(len(val_samples))
print(len(test_samples))

{'video': 'Интервью с Максимом Ларионовым. 2 часть. C субтитрами.mp4', 'start': 194.96, 'end': 205.96, 'frame_ids': [4875, 4878, 4881, 4884, 4887, 4890, 4893, 4896, 4899, 4902, 4905, 4908, 4911, 4914, 4917, 4920, 4923, 4926, 4929, 4932, 4935, 4938, 4941, 4944, 4947, 4950, 4953, 4956, 4959, 4962, 4965, 4968, 4971, 4974, 4977, 4980, 4983, 4986, 4989, 4992, 4995, 4998, 5001, 5004, 5007, 5010, 5013, 5016, 5019, 5022, 5025, 5028, 5031, 5034, 5037, 5040, 5043, 5046, 5049, 5052, 5055, 5058, 5061, 5064, 5067, 5070, 5073, 5076, 5079, 5082, 5085, 5088, 5091, 5094, 5097, 5100, 5103, 5106, 5109, 5112, 5115, 5118, 5121, 5124, 5127, 5130, 5133, 5136, 5139, 5142, 5145, 5148], 'text': 'По моим наблюдениям... Первый молодёжный форум провели в 2012 году в Костроме. Приехало много народу, организовано всё было замечательно… лекции, семинары – всё было замечательно. Результаты были более чем скромными: появилось всего несколько лидеров.', 'split': 'train'}
28818
7205
2633


In [ ]:
vn = 'Уникальная частная коллекция Ивана Бирюкова. История реабилитационной индустрии.С субтитрами.mp4'
frame_ids = []
for ts in train_samples:
    if ts['video'] == vn:
        frame_ids.extend(ts['frame_ids'])
sorted(frame_ids)[500:530]